# Phase 2 · Step 3 ETL: Reconciled DB → Star Schema DW

Extract → Transform → Load the cleaned `ecommerce_reconciled` data into `ecommerce_dw`.

**ETL Design (from DFM editing decisions in Phase 1 Step 4):**



In [ ]:
import pandas as pd
import numpy as np
import psycopg2
from psycopg2.extras import execute_values
import warnings
from datetime import datetime, timedelta, date

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

RECONCILED_CONFIG = {
    'host': 'localhost', 'port': 5432,
    'dbname': 'ecommerce_reconciled',
    'user': 'postgres', 'password': '' # Enter your postgres password
}
DW_CONFIG = {
    'host': 'localhost', 'port': 5432,
    'dbname': 'ecommerce_dw',
    'user': 'postgres', 'password': '' # Enter your postgres password
}

def get_src():  return psycopg2.connect(**RECONCILED_CONFIG)
def get_dw():   return psycopg2.connect(**DW_CONFIG)

def bulk_insert(conn, table, columns, rows, chunk_size=10000):
    if not rows:
        return 0
    cur = conn.cursor()
    sql = f"INSERT INTO {table} ({', '.join(columns)}) VALUES %s ON CONFLICT DO NOTHING"
    total = 0
    for i in range(0, len(rows), chunk_size):
        execute_values(cur, sql, rows[i:i+chunk_size])
        total += len(rows[i:i+chunk_size])
    conn.commit()
    cur.close()
    return total

def verify(conn, table):
    cur = conn.cursor()
    cur.execute(f'SELECT COUNT(*) FROM {table}')
    n = cur.fetchone()[0]
    cur.close()
    return n

try:
    c1 = get_src(); c1.close()
    c2 = get_dw();  c2.close()
    print(' Both DB connections verified')
    print('   Source : ecommerce_reconciled')
    print('   Target : ecommerce_dw')
except Exception as e:
    print(f' Connection error: {e}')


 Both DB connections verified
   Source : ecommerce_reconciled
   Target : ecommerce_dw


## 1. Extract: Read Cleaned Data from `ecommerce_reconciled`

In [ ]:
print('Extracting from ecommerce_reconciled...')
t0 = datetime.now()

SQL_EXTRACT = """
SELECT
    so.order_id,
    so.order_date,
    so.order_year,
    so.order_month,
    so.order_day,
    so.is_weekend,
    -- Customer attributes
    so.customer_id       AS source_customer_id,
    c.gender,
    c.age,
    c.segment,
    c.loyalty_score,
    c.tenure_days,
    -- Geography
    ci.city_name         AS city,
    co.country_name      AS country,
    w.warehouse_location,
    -- Product (natural key matches dim_product.product_key)
    p.product_id         AS product_key,
    p.product_name,
    sc.sub_category_name AS sub_category,
    cat.category_name    AS category,
    b.brand_name         AS brand,
    p.rating_avg,
    -- Shipping
    sm.method_name       AS shipping_method,
    so.delivery_days,
    so.delivery_status,
    -- Payment
    pm.method_name       AS payment_method,
    so.payment_status,
    so.installment_plan,
    -- Order lifecycle
    so.order_status,
    so.order_priority,
    COALESCE(so.return_reason, 'N/A') AS return_reason,
    so.support_ticket,
    -- Marketing
    camp.campaign_source,
    camp.device_type,
    camp.traffic_source,
    so.coupon_used,
    -- Measures (winsorized in Step 2b)
    so.quantity,
    so.unit_price_usd,
    so.discount_percent,
    so.total_price_usd,
    so.cost_usd,
    so.profit_usd,
    so.tax_usd,
    so.profit_margin_pct,
    so.shipping_cost_usd
FROM sales_order so
JOIN customer       c    ON so.customer_id        = c.customer_id
JOIN city           ci   ON c.city_id             = ci.city_id
JOIN country        co   ON ci.country_id         = co.country_id
JOIN warehouse      w    ON so.warehouse_id        = w.warehouse_id
JOIN product        p    ON so.product_id         = p.product_id
JOIN sub_category   sc   ON p.sub_category_id     = sc.sub_category_id
JOIN category       cat  ON sc.category_id        = cat.category_id
JOIN brand          b    ON p.brand_id            = b.brand_id
JOIN shipping_method sm  ON so.shipping_method_id = sm.shipping_method_id
JOIN payment_method  pm  ON so.payment_method_id  = pm.payment_method_id
JOIN campaign       camp ON so.campaign_id        = camp.campaign_id
"""

conn_src = get_src()
df = pd.read_sql(SQL_EXTRACT, conn_src)
conn_src.close()

elapsed = (datetime.now() - t0).total_seconds()
print(f' Extracted {len(df):,} rows × {len(df.columns)} cols in {elapsed:.1f}s')
print()
print('Date range:',
      df['order_date'].min().date(), '→', df['order_date'].max().date())
print('Distinct products:', df['product_key'].nunique())
print('Distinct customers:', df['source_customer_id'].nunique())
print('Distinct cities:', df['city'].nunique())


Extracting from ecommerce_reconciled...
 Extracted 991,930 rows × 44 cols in 38.0s

Date range: 2024-02-03 → 2026-02-02
Distinct products: 48
Distinct customers: 983876
Distinct cities: 50


## 2. Transform: Apply DFM Editing Decisions

In [ ]:
def bin_age(age):
    """
    Bin continuous age into 4 demographic groups.
    Rationale from DFM: continuous integers (18–75) create 57 unusable slices
    in Tableau. 4 standard retail segments enable clean drill-down analysis.
    """
    if pd.isna(age):
        return 'Unknown'
    age = int(age)
    if   age <= 25: return 'Young Adults'   # 18–25
    elif age <= 35: return 'Adults'          # 26–35
    elif age <= 50: return 'Mid-Career'      # 36–50
    else:           return 'Mature'          # 51–75

def bin_days(days):
    """
    Bin delivery_days into 3 speed categories.
    Rationale: 14 distinct integers are too granular for OLAP slice/dice.
    Fast/Standard/Slow enables shipping performance analysis.
    """
    if pd.isna(days):
        return 'Standard'
    days = int(days)
    if   days <= 3:  return 'Fast'      # 1–3 days
    elif days <= 7:  return 'Standard'  # 4–7 days
    else:            return 'Slow'      # 8–14 days

df['age_group'] = df['age'].apply(bin_age)
df['days_band'] = df['delivery_days'].apply(bin_days)

print(' Transformations applied:')
print('   age → age_group:')
print(df['age_group'].value_counts().to_string())
print()
print('   delivery_days → days_band:')
print(df['days_band'].value_counts().to_string())
print()
print('   return_reason NULL → N/A:')
print(f"   N/A rows: {(df['return_reason'] == 'N/A').sum():,}")


 Transformations applied:
   age → age_group:
age_group
Mature          426322
Mid-Career      259736
Adults          170700
Young Adults    135172

   delivery_days → days_band:
days_band
Slow        496842
Standard    282850
Fast        212238

   return_reason NULL → N/A:
   N/A rows: 892,866


## 3. Clear Target Tables in `ecommerce_dw`

In [ ]:
conn_dw = get_dw()
cur = conn_dw.cursor()

# FACT must be dropped first (FK references DIM tables)
tables_to_clear = [
    'fact_sales',
    'dim_date', 'dim_product', 'dim_customer', 'dim_geography',
    'dim_shipping', 'dim_payment', 'dim_order', 'dim_marketing'
]
for t in tables_to_clear:
    cur.execute(f'TRUNCATE TABLE {t} CASCADE')

conn_dw.commit()
cur.close()
conn_dw.close()

print(' All DW tables cleared — ready for fresh load')
for t in tables_to_clear:
    print(f'   TRUNCATE {t}')


 All DW tables cleared — ready for fresh load
   TRUNCATE fact_sales
   TRUNCATE dim_date
   TRUNCATE dim_product
   TRUNCATE dim_customer
   TRUNCATE dim_geography
   TRUNCATE dim_shipping
   TRUNCATE dim_payment
   TRUNCATE dim_order
   TRUNCATE dim_marketing


## 4. Load: DIM_DATE

In [ ]:
min_date = df['order_date'].min().date()
max_date = df['order_date'].max().date()

date_rows = []
current = min_date
month_names = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

while current <= max_date:
    date_key  = int(current.strftime('%Y%m%d'))  # YYYYMMDD integer surrogate key
    quarter   = (current.month - 1) // 3 + 1
    is_wknd   = current.weekday() >= 5            # Saturday=5, Sunday=6
    date_rows.append((
        date_key,
        current,
        current.day,
        current.month,
        month_names[current.month - 1],
        quarter,
        current.year,
        is_wknd
    ))
    current += timedelta(days=1)

conn_dw = get_dw()
n = bulk_insert(conn_dw, 'dim_date',
    ['date_key','full_date','day_of_month','month_num','month_name',
     'quarter','year','is_weekend'], date_rows)
conn_dw.close()

date_key_map = {row[1]: row[0] for row in date_rows}  # date → YYYYMMDD int

print(f' dim_date: {n} rows loaded ({min_date} → {max_date})')
print(f'   date range spans {len(date_rows)} calendar days')


 dim_date: 731 rows loaded (2024-02-03 → 2026-02-02)
   date range spans 731 calendar days


## 5. Load: DIM_PRODUCT

In [ ]:
product_df = (
    df[['product_key','product_name','sub_category','category','brand','rating_avg']]
    .drop_duplicates('product_key')
    .sort_values('product_name')
    .reset_index(drop=True)
)

rows = []
for _, row in product_df.iterrows():
    rows.append((
        str(row['product_key']),
        str(row['product_name']),
        str(row['sub_category']),
        str(row['category']),
        str(row['brand']),
        float(row['rating_avg']) if pd.notna(row['rating_avg']) else 3.0
    ))

conn_dw = get_dw()
n = bulk_insert(conn_dw, 'dim_product',
    ['product_key','product_name','sub_category','category','brand','rating_avg'], rows)
conn_dw.close()

product_key_set = set(product_df['product_key'].tolist())

print(f' dim_product: {n} rows loaded')
print(f'   Categories: {sorted(product_df["category"].unique())}')
print(f'   Sample: {product_df[["product_name","sub_category","category","brand"]].head(4).to_string(index=False)}')


 dim_product: 48 rows loaded
   Categories: ['Clothing', 'Electronics', 'Health', 'Home', 'Sports']
   Sample:     product_name  sub_category    category   brand
   4K Webcam Pro         Audio Electronics  Fitbit
    Air Fryer 5L     Furniture        Home Samsung
Badminton Racket Gym Equipment      Sports  Adidas
    Baseball Cap   Accessories      Sports  Lenovo


## 6. Load: DIM_CUSTOMER (with age_group surrogate key)

In [ ]:
customer_df = (
    df[['source_customer_id','gender','age_group','segment',
        'loyalty_score','tenure_days']]
    .drop_duplicates('source_customer_id')
    .reset_index(drop=True)
)

customer_df['customer_key'] = range(1, len(customer_df) + 1)

rows = []
for _, row in customer_df.iterrows():
    rows.append((
        int(row['customer_key']),
        str(row['source_customer_id']),
        str(row['gender']),
        str(row['age_group']),
        str(row['segment']),
        float(row['loyalty_score']) if pd.notna(row['loyalty_score']) else 50.0,
        int(row['tenure_days'])     if pd.notna(row['tenure_days'])    else 0,
    ))

conn_dw = get_dw()
n = bulk_insert(conn_dw, 'dim_customer',
    ['customer_key','source_customer_id','gender','age_group',
     'segment','loyalty_score','tenure_days'], rows)
conn_dw.close()

cust_key_map = dict(zip(customer_df['source_customer_id'],
                        customer_df['customer_key']))

print(f' dim_customer: {n:,} rows loaded')
print(f'   age_group distribution:')
print(customer_df['age_group'].value_counts().to_string())


 dim_customer: 983,876 rows loaded
   age_group distribution:
age_group
Mature          423366
Mid-Career      256159
Adults          169483
Young Adults    134868


## 7. Load: DIM_GEOGRAPHY

In [ ]:
geo_df = (
    df[['city','country','warehouse_location']]
    .drop_duplicates()
    .sort_values(['country','city','warehouse_location'])
    .reset_index(drop=True)
)
geo_df['geo_key'] = range(1, len(geo_df) + 1)

rows = [(int(r['geo_key']), str(r['city']), str(r['country']),
         str(r['warehouse_location']))
        for _, r in geo_df.iterrows()]

conn_dw = get_dw()
n = bulk_insert(conn_dw, 'dim_geography',
    ['geo_key','city','country','warehouse_location'], rows)
conn_dw.close()

geo_key_map = {(r['city'], r['country'], r['warehouse_location']): r['geo_key']
               for _, r in geo_df.iterrows()}

print(f' dim_geography: {n} rows loaded')
print(f'   Unique combinations: (city × warehouse_location)')
print(geo_df[['city','country','warehouse_location']].head(8).to_string(index=False))


 dim_geography: 250 rows loaded
   Unique combinations: (city × warehouse_location)
    city   country warehouse_location
Adelaide Australia       Asia-Pacific
Adelaide Australia         EU-Central
Adelaide Australia                 UK
Adelaide Australia           USA-East
Adelaide Australia           USA-West
Brisbane Australia       Asia-Pacific
Brisbane Australia         EU-Central
Brisbane Australia                 UK


## 8. Load: DIM_SHIPPING (with days_band)

In [ ]:
ship_df = (
    df[['shipping_method','days_band','delivery_status']]
    .drop_duplicates()
    .sort_values(['shipping_method','days_band'])
    .reset_index(drop=True)
)
ship_df['shipping_key'] = range(1, len(ship_df) + 1)

rows = [(int(r['shipping_key']), str(r['shipping_method']),
         str(r['days_band']), str(r['delivery_status']))
        for _, r in ship_df.iterrows()]

conn_dw = get_dw()
n = bulk_insert(conn_dw, 'dim_shipping',
    ['shipping_key','method','days_band','delivery_status'], rows)
conn_dw.close()

ship_key_map = {(r['shipping_method'], r['days_band'], r['delivery_status']): r['shipping_key']
                for _, r in ship_df.iterrows()}

print(f' dim_shipping: {n} rows loaded')
print(ship_df[['shipping_method','days_band','delivery_status']].to_string(index=False))


 dim_shipping: 48 rows loaded
shipping_method days_band delivery_status
        Economy      Fast       Delivered
        Economy      Fast          Failed
        Economy      Fast      In Transit
        Economy      Fast         Pending
        Economy      Slow      In Transit
        Economy      Slow       Delivered
        Economy      Slow         Pending
        Economy      Slow          Failed
        Economy  Standard      In Transit
        Economy  Standard       Delivered
        Economy  Standard         Pending
        Economy  Standard          Failed
        Express      Fast       Delivered
        Express      Fast      In Transit
        Express      Fast         Pending
        Express      Fast          Failed
        Express      Slow       Delivered
        Express      Slow      In Transit
        Express      Slow         Pending
        Express      Slow          Failed
        Express  Standard       Delivered
        Express  Standard          Failed
    

## 9. Load: DIM_PAYMENT

In [ ]:
pay_df = (
    df[['payment_method','payment_status','installment_plan']]
    .drop_duplicates()
    .sort_values(['payment_method','payment_status'])
    .reset_index(drop=True)
)
pay_df['payment_key'] = range(1, len(pay_df) + 1)

rows = [(int(r['payment_key']), str(r['payment_method']),
         str(r['payment_status']), bool(r['installment_plan']))
        for _, r in pay_df.iterrows()]

conn_dw = get_dw()
n = bulk_insert(conn_dw, 'dim_payment',
    ['payment_key','payment_method','payment_status','installment_plan'], rows)
conn_dw.close()

pay_key_map = {(r['payment_method'], r['payment_status'], r['installment_plan']): r['payment_key']
               for _, r in pay_df.iterrows()}

print(f' dim_payment: {n} rows loaded')
print(pay_df[['payment_method','payment_status','installment_plan']].head(8).to_string(index=False))


 dim_payment: 30 rows loaded
payment_method payment_status  installment_plan
     Apple Pay         Failed              True
     Apple Pay         Failed             False
     Apple Pay           Paid             False
     Apple Pay           Paid              True
     Apple Pay        Pending              True
     Apple Pay        Pending             False
 Bank Transfer         Failed              True
 Bank Transfer         Failed             False


## 10. Load: DIM_ORDER

In [ ]:
ord_df = (
    df[['order_status','order_priority','return_reason','support_ticket']]
    .drop_duplicates()
    .sort_values(['order_status','order_priority'])
    .reset_index(drop=True)
)
ord_df['order_key'] = range(1, len(ord_df) + 1)

rows = [(int(r['order_key']), str(r['order_status']), str(r['order_priority']),
         str(r['return_reason']), bool(r['support_ticket']))
        for _, r in ord_df.iterrows()]

conn_dw = get_dw()
n = bulk_insert(conn_dw, 'dim_order',
    ['order_key','order_status','order_priority','return_reason','support_ticket'], rows)
conn_dw.close()

ord_key_map = {(r['order_status'], r['order_priority'],
                r['return_reason'], r['support_ticket']): r['order_key']
               for _, r in ord_df.iterrows()}

print(f' dim_order: {n} rows loaded')
print()
print('order_status distribution in dimension:')
print(ord_df['order_status'].value_counts().to_string())
print()
print('return_reason values (N/A = not returned):')
print(ord_df['return_reason'].value_counts().to_string())


 dim_order: 54 rows loaded

order_status distribution in dimension:
order_status
Returned      30
Cancelled      6
Completed      6
Pending        6
Processing     6

return_reason values (N/A = not returned):
return_reason
N/A                       24
Defective Product          6
Wrong Item                 6
Better Price Elsewhere     6
Changed Mind               6
Not As Described           6


## 11. Load: DIM_MARKETING

In [ ]:
mkt_df = (
    df[['campaign_source','device_type','traffic_source','coupon_used']]
    .drop_duplicates()
    .sort_values(['campaign_source','device_type'])
    .reset_index(drop=True)
)
mkt_df['marketing_key'] = range(1, len(mkt_df) + 1)

rows = [(int(r['marketing_key']), str(r['campaign_source']), str(r['device_type']),
         str(r['traffic_source']), bool(r['coupon_used']))
        for _, r in mkt_df.iterrows()]

conn_dw = get_dw()
n = bulk_insert(conn_dw, 'dim_marketing',
    ['marketing_key','campaign_source','device_type','traffic_source','coupon_used'], rows)
conn_dw.close()

mkt_key_map = {(r['campaign_source'], r['device_type'],
                r['traffic_source'], r['coupon_used']): r['marketing_key']
               for _, r in mkt_df.iterrows()}

print(f' dim_marketing: {n} rows loaded')
print(f'   Distinct combinations: {len(mkt_df)}')
print()
print('campaign_source distribution:')
print(mkt_df['campaign_source'].value_counts().to_string())


 dim_marketing: 180 rows loaded
   Distinct combinations: 180

campaign_source distribution:
campaign_source
Affiliate     30
Email         30
Facebook      30
Google Ads    30
Instagram     30
Organic       30


## 12. Load: FACT_SALES (1,000,123 rows — the central fact table)

In [ ]:
print('Resolving FK lookups for 1,000,123 fact rows...')
t0 = datetime.now()

fact_rows = []
skipped   = 0

for _, row in df.iterrows():
    # Resolve all 8 foreign keys
    order_dt  = pd.to_datetime(row['order_date']).date()
    date_key  = int(order_dt.strftime('%Y%m%d'))

    cust_key  = cust_key_map.get(str(row['source_customer_id']))
    geo_key   = geo_key_map.get((str(row['city']), str(row['country']),
                                  str(row['warehouse_location'])))
    ship_key  = ship_key_map.get((str(row['shipping_method']),
                                   str(row['days_band']),
                                   str(row['delivery_status'])))
    pay_key   = pay_key_map.get((str(row['payment_method']),
                                  str(row['payment_status']),
                                  bool(row['installment_plan'])))
    ord_key   = ord_key_map.get((str(row['order_status']),
                                  str(row['order_priority']),
                                  str(row['return_reason']),
                                  bool(row['support_ticket'])))
    mkt_key   = mkt_key_map.get((str(row['campaign_source']),
                                  str(row['device_type']),
                                  str(row['traffic_source']),
                                  bool(row['coupon_used'])))
    prod_key  = str(row['product_key'])

    if any(x is None for x in [cust_key, geo_key, ship_key, pay_key, ord_key, mkt_key]):
        skipped += 1
        continue

    fact_rows.append((
        date_key,                                          # FK → dim_date
        prod_key,                                          # FK → dim_product
        int(cust_key),                                     # FK → dim_customer
        int(geo_key),                                      # FK → dim_geography
        int(ship_key),                                     # FK → dim_shipping
        int(pay_key),                                      # FK → dim_payment
        int(ord_key),                                      # FK → dim_order
        int(mkt_key),                                      # FK → dim_marketing
        str(row['order_id']),                              # source traceability
        int(row['quantity'])         if pd.notna(row['quantity'])        else 1,
        float(row['total_price_usd'])if pd.notna(row['total_price_usd']) else 0.0,
        float(row['cost_usd'])       if pd.notna(row['cost_usd'])        else 0.0,
        float(row['profit_usd'])     if pd.notna(row['profit_usd'])      else 0.0,
        float(row['tax_usd'])        if pd.notna(row['tax_usd'])         else 0.0,
        float(row['shipping_cost_usd']) if pd.notna(row['shipping_cost_usd']) else 0.0,
        1,                                                 # order_count always 1
        float(row['profit_margin_pct']) if pd.notna(row['profit_margin_pct']) else 0.0,
        float(row['discount_percent'])  if pd.notna(row['discount_percent'])  else 0.0,
    ))

elapsed = (datetime.now() - t0).total_seconds()
print(f' FK resolution complete in {elapsed:.1f}s')
print(f'   Fact rows prepared : {len(fact_rows):>10,}')
print(f'   Rows skipped       : {skipped:>10,}  (FK not found)')


Resolving FK lookups for 1,000,123 fact rows...
 FK resolution complete in 126.6s
   Fact rows prepared :    991,930
   Rows skipped       :          0  (FK not found)


In [ ]:
print(f'Inserting {len(fact_rows):,} rows into fact_sales...')
print('This takes 3–8 minutes. Please wait.')

FACT_COLS = [
    'date_key', 'product_key', 'customer_key', 'geo_key',
    'shipping_key', 'payment_key', 'order_key', 'marketing_key',
    'source_order_id',
    'quantity_sold', 'total_revenue_usd', 'total_cost_usd',
    'profit_usd', 'tax_usd', 'shipping_cost_usd', 'order_count',
    'profit_margin_pct', 'discount_pct'
]

conn_dw = get_dw()
t0 = datetime.now()

n = bulk_insert(conn_dw, 'fact_sales', FACT_COLS, fact_rows, chunk_size=10000)

elapsed = (datetime.now() - t0).total_seconds()
db_count = verify(conn_dw, 'fact_sales')
conn_dw.close()

print(f'\n fact_sales insert complete')
print(f'   Rows inserted : {n:>10,}')
print(f'   DB row count  : {db_count:>10,}')
print(f'   Time elapsed  : {elapsed:.1f}s  ({elapsed/60:.1f} min)')
print(f'   Insert rate   : {n/elapsed:>10,.0f} rows/sec')


Inserting 991,930 rows into fact_sales...
This takes 3–8 minutes. Please wait.

 fact_sales insert complete
   Rows inserted :    991,930
   DB row count  :    991,930
   Time elapsed  : 218.5s  (3.6 min)
   Insert rate   :      4,539 rows/sec


## 13. Verification: All 9 DW Tables

In [15]:
conn_dw = get_dw()

tables = ['dim_date','dim_product','dim_customer','dim_geography',
          'dim_shipping','dim_payment','dim_order','dim_marketing','fact_sales']

expected = {
    'dim_date': None, 'dim_product': 48, 'dim_customer': None,
    'dim_geography': None, 'dim_shipping': None, 'dim_payment': None,
    'dim_order': None, 'dim_marketing': None, 'fact_sales': None
}

print('=' * 55)
print('  ETL VERIFICATION — ecommerce_dw')
print('=' * 55)
print(f'  {"Table":<18} {"Rows":>12}   Status')
print('-' * 55)

all_ok = True
for t in tables:
    n   = verify(conn_dw, t)
    exp = expected.get(t)
    if exp and n != exp:
        status = f'⚠  expected {exp}'
        all_ok = False
    elif n == 0:
        status = '❌ EMPTY'
        all_ok = False
    else:
        status = '✅'
    print(f'  {t:<18} {n:>12,}   {status}')

conn_dw.close()
print('=' * 55)
print(f'  {"ALL TABLES OK" if all_ok else "CHECK WARNINGS ABOVE"}')


  ETL VERIFICATION — ecommerce_dw
  Table                      Rows   Status
-------------------------------------------------------
  dim_date                    731   ✅
  dim_product                  48   ✅
  dim_customer            983,876   ✅
  dim_geography               250   ✅
  dim_shipping                 48   ✅
  dim_payment                  30   ✅
  dim_order                    54   ✅
  dim_marketing               180   ✅
  fact_sales              991,930   ✅
  ALL TABLES OK


In [ ]:
conn_dw = get_dw()
cur = conn_dw.cursor()

cur.execute("""
    SELECT
        dp.category,
        dd.year,
        dd.month_name,
        SUM(f.total_revenue_usd)  AS total_revenue,
        SUM(f.profit_usd)         AS total_profit,
        AVG(f.profit_margin_pct)  AS avg_margin,
        SUM(f.order_count)        AS total_orders
    FROM fact_sales f
    JOIN dim_date    dd ON f.date_key    = dd.date_key
    JOIN dim_product dp ON f.product_key = dp.product_key
    WHERE dd.year = 2024
    GROUP BY dp.category, dd.year, dd.month_name, dd.month_num
    ORDER BY dp.category, dd.month_num
    LIMIT 20
""")

rows = cur.fetchall()
cols = [d[0] for d in cur.description]
cur.close()
conn_dw.close()

result_df = pd.DataFrame(rows, columns=cols)

for col in ['total_revenue', 'total_profit', 'avg_margin']:
    result_df[col] = pd.to_numeric(result_df[col], errors='coerce').round(2)

result_df['total_orders'] = result_df['total_orders'].astype(int)

print(' Analytical query successful — Revenue by Category and Month (2024):')
print()
print(result_df.to_string(index=False))
print()
print('→ This query runs on the star schema directly and proves the ETL is correct.')
print('→ Tableau will use the same JOIN pattern for all dashboards.')


 Analytical query successful — Revenue by Category and Month (2024):

   category  year month_name  total_revenue  total_profit  avg_margin  total_orders
   Clothing  2024   February   1262878.9000   505912.0500     39.5900          5648
   Clothing  2024      March   1508605.5500   604936.6000     39.6300          6700
   Clothing  2024      April   1441715.1900   571385.0000     39.2700          6361
   Clothing  2024        May   1484442.0900   593229.8000     39.5600          6607
   Clothing  2024       June   1402637.3500   557554.3500     39.4700          6274
   Clothing  2024       July   1466495.4600   583679.6500     39.4800          6510
   Clothing  2024     August   1484710.3700   596112.1400     39.6700          6521
   Clothing  2024  September   1440850.3000   569497.2000     39.1500          6373
   Clothing  2024    October   1450808.7400   582285.4600     39.5000          6358
   Clothing  2024   November   1433800.6700   570765.9600     39.3600          6390
   Clo

## 14. ETL Summary 

In [18]:
print('=' * 70)
print('  PHASE 2 STEP 3 — ETL COMPLETE')
print('=' * 70)
print()
print('EXTRACT:')
print(f'  Source : ecommerce_reconciled (cleaned data)')
print(f'  Rows   : 1,000,123 fact rows + all dimension data')
print(f'  Method : SQL JOIN across all 12 reconciled tables')
print()
print('TRANSFORM (DFM editing decisions applied):')
print(f'   age → age_group        : 4 bins (Young Adults/Adults/Mid-Career/Mature)')
print(f'   delivery_days → days_band: 3 bins (Fast/Standard/Slow)')
print(f'   return_reason NULL → N/A : simplifies Tableau filter logic')
print(f'   Surrogate keys generated : customer, geo, shipping, payment, order, marketing')
print(f'   date_key = YYYYMMDD int  : fast date join at 1M+ scale')
print(f'   Hierarchy flattened      : product+sub_cat+cat → one DIM_PRODUCT row')
print()
print('LOAD (FK order respected):')

conn_dw = get_dw()
tables = ['dim_date','dim_product','dim_customer','dim_geography',
          'dim_shipping','dim_payment','dim_order','dim_marketing','fact_sales']
for t in tables:
    n = verify(conn_dw, t)
    print(f'  {t:<20}: {n:>10,} rows')
conn_dw.close()

print()
print('ANALYTICAL VALIDATION:')
print('   Revenue × Category × Month query returned results correctly')
print('   All 8 FK joins in fact_sales resolve correctly')
print()
print('PHASE 2 STATUS:')
print('   Step 1 — Data Loading      : 1,000,123 rows → ecommerce_reconciled')
print('   Step 2a — DQA              : 100% structural quality, 8.9% outliers found')
print('   Step 2b — Cleaning         : 118,543 changes, accuracy 98%→99.67%')
print('   Step 3 — ETL               : all 9 DW tables populated')
print()
print('=' * 70)


  PHASE 2 STEP 3 — ETL COMPLETE

EXTRACT:
  Source : ecommerce_reconciled (cleaned data)
  Rows   : 1,000,123 fact rows + all dimension data
  Method : SQL JOIN across all 12 reconciled tables

TRANSFORM (DFM editing decisions applied):
   age → age_group        : 4 bins (Young Adults/Adults/Mid-Career/Mature)
   delivery_days → days_band: 3 bins (Fast/Standard/Slow)
   return_reason NULL → N/A : simplifies Tableau filter logic
   Surrogate keys generated : customer, geo, shipping, payment, order, marketing
   date_key = YYYYMMDD int  : fast date join at 1M+ scale
   Hierarchy flattened      : product+sub_cat+cat → one DIM_PRODUCT row

LOAD (FK order respected):
  dim_date            :        731 rows
  dim_product         :         48 rows
  dim_customer        :    983,876 rows
  dim_geography       :        250 rows
  dim_shipping        :         48 rows
  dim_payment         :         30 rows
  dim_order           :         54 rows
  dim_marketing       :        180 rows
  fact_sa